# Failure Case Analysis

This notebook visualizes misclassified images from the test set.
It helps identify systematic error patterns (e.g., long hair on males,
ambiguous poses) that can be discussed in the Experiments section of the report.

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from src.data.dataset import CelebAGenderDataset
from src.data.transforms import get_eval_transforms
from src.models.factory import build_model
from src.utils.config import load_config, get_device

%matplotlib inline

## 1. Configuration

Set the model checkpoint and config to analyze.

In [ ]:
CHECKPOINT_PATH = "../checkpoints/best_model.pt"
CONFIG_PATH = "../configs/custom_cnn.yaml"  # change for each model

config = load_config(CONFIG_PATH)
device = get_device(config["device"])

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model = build_model(config)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print(f"Model: {config['model']['name']}")
print(f"Val metrics from checkpoint: {checkpoint.get('val_metrics', 'N/A')}")

## 2. Run inference on test set

In [ ]:
data_cfg = config["data"]
test_dataset = CelebAGenderDataset(
    root_dir=data_cfg["root_dir"],
    split="test",
    transform=get_eval_transforms(data_cfg["image_size"]),
)

# Also keep a version without transforms for visualization
test_dataset_raw = CelebAGenderDataset(
    root_dir=data_cfg["root_dir"],
    split="test",
    transform=None,
)

loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in loader:
        logits = model(images.to(device)).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

probs = np.concatenate(all_probs)
labels = np.concatenate(all_labels)
preds = (probs >= 0.5).astype(int)

misclassified = np.where(preds != labels)[0]
print(f"Misclassified: {len(misclassified)} / {len(labels)} ({100*len(misclassified)/len(labels):.1f}%)")

## 3. Visualize misclassified images

In [ ]:
GENDER_MAP = {0: "Female", 1: "Male"}
N_SHOW = 20

# Sort by confidence (most confident wrong predictions first)
confidence = np.abs(probs[misclassified] - 0.5)
sorted_idx = misclassified[np.argsort(-confidence)][:N_SHOW]

cols = 5
rows = (N_SHOW + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.5 * rows))
axes = axes.flatten()

for i, idx in enumerate(sorted_idx):
    img, _ = test_dataset_raw[idx]
    axes[i].imshow(img)
    true_label = GENDER_MAP[labels[idx]]
    pred_label = GENDER_MAP[preds[idx]]
    prob = probs[idx]
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label} ({prob:.2f})", fontsize=8)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Top {N_SHOW} Most Confident Misclassifications — {config['model']['name']}", fontsize=12)
plt.tight_layout()
plt.savefig(f"../outputs/{config['model']['name']}_failure_cases.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Error distribution by predicted probability

Shows how prediction confidence distributes for correct vs incorrect predictions.

In [ ]:
correct_mask = preds == labels

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(probs[correct_mask], bins=50, alpha=0.6, label="Correct", color="#4C72B0")
ax.hist(probs[~correct_mask], bins=50, alpha=0.6, label="Misclassified", color="#C44E52")
ax.axvline(0.5, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("Predicted Probability (Male)")
ax.set_ylabel("Count")
ax.set_title(f"Prediction Confidence Distribution — {config['model']['name']}")
ax.legend()
plt.tight_layout()
plt.savefig(f"../outputs/{config['model']['name']}_confidence_dist.png", dpi=150, bbox_inches="tight")
plt.show()